# RL Experiment 03: Curriculum Learning

Trains PPO in staged phases with increasing episode length.


In [ ]:
from schedule_engine.notebooks import (
    build_notebook_config,
    create_env,
    evaluate_agent,
    load_context,
    set_global_seed,
    train_agent,
)

SEED = 42
POP_SIZE = 20

stages = [
    {"name": "easy", "max_generations": 30, "max_steps": 10, "timesteps": 3000},
    {"name": "medium", "max_generations": 50, "max_steps": 15, "timesteps": 4000},
    {"name": "hard", "max_generations": 80, "max_steps": 20, "timesteps": 5000},
]

set_global_seed(SEED)
config = build_notebook_config(seed=SEED, overrides={"pop_size": POP_SIZE})
_, context = load_context("data", config)

agent = None
for stage in stages:
    env = create_env(
        context=context,
        pop_size=POP_SIZE,
        max_generations=stage["max_generations"],
        max_steps=stage["max_steps"],
    )
    if agent is None:
        agent, train_time = train_agent(
            "ppo",
            env,
            timesteps=stage["timesteps"],
            seed=SEED,
        )
    else:
        agent.set_env(env)
        agent.learn(total_timesteps=stage["timesteps"], progress_bar=False)
        train_time = 0.0

    result = evaluate_agent(agent, env, max_generations=stage["max_generations"])
    print(
        f"Stage {stage['name']}: best={result.best_fitness}, conv={result.convergence_gen}" 
        f" (train_time={train_time:.2f}s)"
    )
